# Pipeline Orchestration & Reliability

## Objective

In the previous notebooks, I built a Snowflake analytical warehouse, dbt transformations, customer feature marts, uplift models, a decision engine, and a containerized inference API.

Until now, rebuilding the warehouse depended on manually running commands in the correct order.

In this notebook, I introduce Apache Airflow to coordinate the existing dbt workflow.

The DAG checks Snowflake connectivity, validates the fixed X5 source snapshot, runs the dbt build, and verifies the structured execution results.

I intentionally keep this workflow manually triggered. The project currently uses a fixed historical dataset, and automatic ingestion, unattended authentication, model retraining, and deployment have not yet been implemented.

The objective is to demonstrate repeatable workflow execution, dependency management, failure visibility, and basic operational reliability without duplicating the transformation logic already owned by dbt.

## 1. Orchestration architecture

Airflow and dbt have different responsibilities.

Airflow coordinates the overall workflow, tracks task execution, and prevents downstream tasks from running when prerequisites fail.

dbt remains responsible for SQL transformations, dependency resolution between models, and data-quality tests.

The workflow is:

1. Confirm Snowflake connectivity.
2. Validate the five fixed X5 source-table populations.
3. Execute the existing dbt project.
4. Inspect dbt's structured execution results.

The DAG does not ingest new source files, retrain machine-learning models, modify MLflow records, or redeploy the API.

This separation prevents a routine warehouse rebuild from silently changing the model currently served to applications.

In [1]:
# ============================================================
# Inspect the latest dbt execution
#
# Airflow controls the workflow.
# dbt records the details of individual model/test execution.
#
# This cell reads the existing results; it does not trigger
# another expensive warehouse rebuild.
# ============================================================

from collections import Counter
from pathlib import Path
import json
import sys

import pandas as pd


PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "dbt" / "dbt_project.yml").exists()
)

sys.path.insert(
    0,
    str(PROJECT_ROOT),
)

from scripts.verify_dbt_build import (
    validate_run_results,
)


RESULTS_PATH = (
    PROJECT_ROOT
    / "dbt"
    / "target"
    / "run_results.json"
)


payload = json.loads(
    RESULTS_PATH.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# Apply the SAME verification logic used by Airflow.
# We do not maintain two different sets of acceptance rules.
# ------------------------------------------------------------

summary = validate_run_results(
    payload
)


# ------------------------------------------------------------
# Create a readable execution report.
# ------------------------------------------------------------

execution_report = pd.DataFrame(
    [
        {
            "resource": result.get("unique_id"),
            "status": result.get("status"),
            "execution_seconds":
                result.get("execution_time"),
        }
        for result in payload["results"]
    ]
)


print("Build verification:", summary)

display(
    execution_report.sort_values(
        "execution_seconds",
        ascending=False,
    ).head(15)
)

Build verification: {'total_resources': 64, 'statuses': {'success': 17, 'pass': 47}, 'required_models_verified': 3}


,resource,status,execution_seconds
28,model.retail_growth.fact_transaction_item,success,26.385620
19,model.retail_growth.fact_transaction,success,20.467480
48,model.retail_growth.mart_customer_features,success,14.041524
44,model.retail_growth.mart_product_performance,success,11.263585
40,model.retail_growth.mart_customer_360,success,5.310232
6,model.retail_growth.dim_customer,success,5.139105
39,test.retail_growth.relationships_fact_transact...,pass,4.562591
29,model.retail_growth.mart_customer_activity,success,4.137373
38,test.retail_growth.assert_fact_transaction_ite...,pass,3.883942
26,model.retail_growth.mart_purchase_daily,success,3.569171


## 2. Conclusions and limitations

I introduced Apache Airflow to coordinate the existing Snowflake/dbt warehouse and customer-feature workflow.

The DAG validates the fixed X5 source snapshot before rebuilding downstream models, then inspects dbt's structured execution results to confirm that the expected modeling marts were built successfully.

I reused the existing dbt transformations and data-quality tests rather than recreating their logic inside Airflow.

The workflow is manually triggered and prevents downstream execution when an upstream task fails. I also added unit tests for the execution-result checker to verify that it detects unsuccessful resources and missing required models.

### Key findings

- DAG tasks: **4 — all passed**
- Source tables covered by the snapshot contract: **5**
- Required modeling marts verified: **3**
- Total dbt resources executed: **64**
- Successful dbt resource executions: **17**
- Passing dbt tests: **47**
- Pipeline verification unit tests: **3 passed**
- Full Python test suite: **11 passed**
- Airflow DAG run status: **Success**

### Limitations

This is a local orchestration demonstration, not a fully automated production pipeline.

The DAG rebuilds the existing analytical warehouse from the fixed X5 snapshot. It does not detect or ingest new S3 objects, implement incremental transformations, retrain ML models, register replacement models, or deploy the API.

Airflow and dbt currently execute on my local development machine. A production implementation would require a continuously available orchestration environment, appropriate service authentication, secret management, operational monitoring, and a defined failure-recovery process.

The next stage is to extend this system with new-data processing and more comprehensive reliability controls.